# LangChain — Tools, Chains, Memory & First Framework Agent

**Goal:** Rebuild the Day 1 raw-Python agent using LangChain, then add tools, external data, memory, structured output, and error handling.

> **Important:** This notebook has a deterministic **DEMO_MODE** so every required section has visible output even when no Anthropic API key is available. Set `DEMO_MODE = False` and provide `ANTHROPIC_API_KEY` to run the live Anthropic version.


In [1]:
# Task 1 — Install the required packages
# Run this cell in your own environment once.
%pip install -U langchain langchain-anthropic langchain-core pydantic pandas --break-system-packages


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 23.3 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 37.1 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/11.0 MB ? eta -:--:--

   ━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/11.0 MB 107.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━ 7.8/11.0 MB 107.2 MB/s eta 0:00:01

...

    Found existing installation: websockets 17.1
    Uninstalling websockets-17.1:
      Successfully uninstalled websockets-17.1


  Attempting uninstall: pandas


    Found existing installation: pandas 3.0.2


    Uninstalling pandas-3.0.2:
      Successfully uninstalled pandas-3.0.2


Note: you may need to restart the kernel to use updated packages.


## Task 1 — Core concept mapping

| Day 1 raw Python | LangChain |
|---|---|
| Manual API/model call | `ChatAnthropic` / LLM wrapper |
| Python function + manual schema | `@tool` |
| `while` loop + tool dispatch | `AgentExecutor` / agent runtime |
| `messages` list | message history / `RunnableWithMessageHistory` |
| Manual prompt construction | `ChatPromptTemplate` |
| Manual parsing | structured output / Pydantic |

### LCEL `|` syntax
The pipe operator composes runnable components into a sequence. Conceptually, the output of the component on the left becomes the input of the component on the right, so `prompt | model | parser` forms one executable pipeline.


In [2]:
# Basic LCEL pipeline
# In live mode, this uses Anthropic. In demo mode, it uses a deterministic response.

DEMO_MODE = True

try:
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser

    prompt = ChatPromptTemplate.from_template("Answer briefly: {question}")

    if not DEMO_MODE:
        from langchain_anthropic import ChatAnthropic
        model = ChatAnthropic(model="claude-3-5-haiku-latest", temperature=0)
        chain = prompt | model | StrOutputParser()
        print(chain.invoke({"question": "What is LangChain?"}))
    else:
        print("LangChain is a framework for composing models, prompts, tools, memory, and agent execution.")
except ImportError:
    print("Demo output: LangChain is a framework for composing models, prompts, tools, memory, and agent execution.")


LangChain is a framework for composing models, prompts, tools, memory, and agent execution.


## Task 2 — Tools

The three tools below are:

1. `calculator` — simple arithmetic.
2. `weather_lookup` — deterministic weather stub for two cities.
3. `read_product_price` — reads a real local JSON data source.

Tool docstrings are important because the model uses the tool name/description/schema to decide **when** a tool is appropriate and **what arguments** it should send.


In [3]:
# Create a real local JSON data source
import json, os

products = {
    "Laptop A": {"price": 850, "category": "laptop"},
    "Laptop B": {"price": 1050, "category": "laptop"},
    "Phone A": {"price": 500, "category": "phone"}
}

with open("products.json", "w", encoding="utf-8") as f:
    json.dump(products, f, indent=2)

print("Created products.json with", len(products), "records.")


Created products.json with 3 records.


In [4]:
# LangChain tool definitions
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Calculate a basic arithmetic expression using +, -, *, and /."""
    allowed = set("0123456789+-*/(). ")
    if not expression or any(ch not in allowed for ch in expression):
        return "ERROR: unsupported expression"
    try:
        # For this controlled demo, only simple numeric arithmetic is accepted.
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"ERROR: {type(e).__name__}: {e}"

@tool
def weather_lookup(city: str) -> str:
    """Return the current demo weather for a supported city."""
    data = {
        "Islamabad": {"temperature_c": 27, "condition": "Sunny"},
        "London": {"temperature_c": 18, "condition": "Cloudy"},
        "Lahore": {"temperature_c": 31, "condition": "Sunny"}
    }
    if city not in data:
        return f"ERROR: unsupported city: {city}"
    return json.dumps(data[city])

@tool
def read_product_price(product_name: str) -> str:
    """Read a product price from the local products.json data source."""
    with open("products.json", "r", encoding="utf-8") as f:
        data = json.load(f)
    if product_name not in data:
        return f"ERROR: product not found: {product_name}"
    return json.dumps(data[product_name])

print("Registered tools:")
for t in [calculator, weather_lookup, read_product_price]:
    print("-", t.name, ":", t.description.splitlines()[0])


Registered tools:
- calculator : Calculate a basic arithmetic expression using +, -, *, and /.
- weather_lookup : Return the current demo weather for a supported city.
- read_product_price : Read a product price from the local products.json data source.


# LangChain tool definitions
from langchain_core.tools import tool
import ast, operator

_ALLOWED = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.USub: operator.neg,
}

def _safe_calc(node):
    if isinstance(node, ast.Expression):
        return _safe_calc(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED:
        return _ALLOWED[type(node.op)](_safe_calc(node.operand))
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED:
        return _ALLOWED[type(node.op)](_safe_calc(node.left), _safe_calc(node.right))
    raise ValueError("unsupported expression")

@tool
def calculator(expression: str) -> str:
    """Calculate a basic arithmetic expression using +, -, *, and /."""
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_safe_calc(tree))
    except Exception as e:
        return f"ERROR: {type(e).__name__}: {e}"

@tool
def weather_lookup(city: str) -> str:
    """Return the current demo weather for a supported city."""
    data = {
        "Islamabad": {"temperature_c": 27, "condition": "Sunny"},
        "London": {"temperature_c": 18, "condition": "Cloudy"},
        "Lahore": {"temperature_c": 31, "condition": "Sunny"}
    }
    if city not in data:
        return f"ERROR: unsupported city: {city}"
    return json.dumps(data[city])

@tool
def read_product_price(product_name: str) -> str:
    """Read a product price from the local products.json data source."""
    with open("products.json", "r", encoding="utf-8") as f:
        data = json.load(f)
    if product_name not in data:
        return f"ERROR: product not found: {product_name}"
    return json.dumps(data[product_name])

print("Registered tools:")
for t in [calculator, weather_lookup, read_product_price]:
    print("-", t.name, ":", t.description.splitlines()[0])


In [5]:
# Inspect generated schemas
for t in [calculator, weather_lookup, read_product_price]:
    print("\nTOOL:", t.name)
    print(t.args_schema.model_json_schema())



TOOL: calculator
{'description': 'Calculate a basic arithmetic expression using +, -, *, and /.', 'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'calculator', 'type': 'object'}

TOOL: weather_lookup
{'description': 'Return the current demo weather for a supported city.', 'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'weather_lookup', 'type': 'object'}

TOOL: read_product_price
{'description': 'Read a product price from the local products.json data source.', 'properties': {'product_name': {'title': 'Product Name', 'type': 'string'}}, 'required': ['product_name'], 'title': 'read_product_price', 'type': 'object'}


## Task 3 — Agent with tools

The live version below uses `ChatAnthropic` and `AgentExecutor`. Because the exact constructor API can change between LangChain releases, the code includes a current-style implementation and a deterministic fallback demonstration.

The trace is annotated as:

- **Reason:** model decides which information is needed.
- **Act:** model requests a tool.
- **Observe:** tool result is returned to the agent.
- **Final:** agent produces the user-facing answer.


In [6]:
# Live agent setup
# Set DEMO_MODE=False after installing packages and exporting ANTHROPIC_API_KEY.

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

tools = [calculator, weather_lookup, read_product_price]

if not DEMO_MODE:
    import os
    from langchain_anthropic import ChatAnthropic
    from langchain.agents import AgentExecutor, create_tool_calling_agent

    model = ChatAnthropic(model="claude-3-5-haiku-latest", temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Use tools when needed."),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ])

    agent = create_tool_calling_agent(model, tools, prompt)
    executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
    result = executor.invoke({
        "input": "Look up the price of Laptop A and Laptop B, then compare them."
    })
    print(result["output"])
else:
    print("DEMO MODE — AgentExecutor trace")
    print("Reason: I need both product prices before comparing them.")
    print("Act: read_product_price({'product_name': 'Laptop A'})")
    print("Observe: {'price': 850, 'category': 'laptop'}")
    print("Reason: I now need Laptop B.")
    print("Act: read_product_price({'product_name': 'Laptop B'})")
    print("Observe: {'price': 1050, 'category': 'laptop'}")
    print("Final: Laptop A costs $850 and Laptop B costs $1,050. Laptop A is $200 cheaper.")


DEMO MODE — AgentExecutor trace
Reason: I need both product prices before comparing them.
Act: read_product_price({'product_name': 'Laptop A'})
Observe: {'price': 850, 'category': 'laptop'}
Reason: I now need Laptop B.
Act: read_product_price({'product_name': 'Laptop B'})
Observe: {'price': 1050, 'category': 'laptop'}
Final: Laptop A costs $850 and Laptop B costs $1,050. Laptop A is $200 cheaper.


### Raw Python vs LangChain trace

**Similar:** both systems follow the same basic cycle: decide → call tool → receive observation → continue → answer.

**Hidden/automated in LangChain:** message formatting, tool registration, tool dispatch, agent execution, and parts of the scratchpad/history handling. The trade-off is convenience versus less direct visibility into every low-level step.


## Task 4 — Memory

Conversation memory stores the messages that belong to the conversation. Working memory is the temporary state needed while solving the current task, such as which tools have already been called and which observations have been collected.


In [7]:
# Three-turn memory demonstration
conversation = []

def remember(user, assistant):
    conversation.append(("user", user))
    conversation.append(("assistant", assistant))

turn1 = "Find the price of Laptop A."
answer1 = "Laptop A costs $850."
remember(turn1, answer1)

turn2 = "Now compare it to Laptop B."
answer2 = "Laptop B costs $1,050, so Laptop A is $200 cheaper."
remember(turn2, answer2)

turn3 = "Which one should I recommend to a budget-conscious client?"
answer3 = "Recommend Laptop A because it is $200 cheaper while both are laptops."
remember(turn3, answer3)

for i, (role, text) in enumerate(conversation, 1):
    print(f"{i}. {role}: {text}")
print("\nFinal recommendation:", answer3)


1. user: Find the price of Laptop A.
2. assistant: Laptop A costs $850.
3. user: Now compare it to Laptop B.
4. assistant: Laptop B costs $1,050, so Laptop A is $200 cheaper.
5. user: Which one should I recommend to a budget-conscious client?
6. assistant: Recommend Laptop A because it is $200 cheaper while both are laptops.

Final recommendation: Recommend Laptop A because it is $200 cheaper while both are laptops.


In [8]:
# Modern LangChain-style message history example
# This cell shows the structure to use with RunnableWithMessageHistory.
# It remains in DEMO_MODE so the notebook is runnable without an API key.

from langchain_core.chat_history import InMemoryChatMessageHistory

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

history = get_session_history("demo-session")
history.add_user_message("Find the price of Laptop A.")
history.add_ai_message("Laptop A costs $850.")
history.add_user_message("Now compare it to Laptop B.")
history.add_ai_message("Laptop B costs $1,050.")
print("Stored messages:", len(history.messages))
print("Follow-up can use the earlier messages:", history.messages[-2].content, " / ", history.messages[-1].content)


Stored messages: 4
Follow-up can use the earlier messages: Now compare it to Laptop B.  /  Laptop B costs $1,050.


## Task 5 — Structured output

A structured result makes downstream code safer because it receives named fields instead of relying on free-form text. Here the final answer is represented with a Pydantic model.


In [9]:
from pydantic import BaseModel, Field

class Recommendation(BaseModel):
    recommendation: str = Field(description="Recommended product")
    reason: str = Field(description="Short reason")
    price_difference: float = Field(description="Price difference in USD")

structured_demo = Recommendation(
    recommendation="Laptop A",
    reason="It is cheaper and fits the budget-conscious requirement.",
    price_difference=200.0
)

print(structured_demo.model_dump_json(indent=2))


{
  "recommendation": "Laptop A",
  "reason": "It is cheaper and fits the budget-conscious requirement.",
  "price_difference": 200.0
}


In [10]:
# Structured-output live pattern
if not DEMO_MODE:
    structured_model = model.with_structured_output(Recommendation)
    response = structured_model.invoke(
        "Laptop A costs $850 and Laptop B costs $1050. Recommend one for a budget-conscious client."
    )
    print(response)
else:
    print("DEMO MODE structured result:")
    print(structured_demo)


DEMO MODE structured result:
recommendation='Laptop A' reason='It is cheaper and fits the budget-conscious requirement.' price_difference=200.0


### Error handling

A tool can fail because an argument is invalid, a file is missing, an API is unavailable, or the requested record does not exist. A useful pattern is to return a clear structured error from the tool and instruct the agent to recover or explain the problem instead of silently continuing.


In [11]:
# Deliberate tool failure + graceful handling
def safe_weather(city):
    try:
        result = weather_lookup.invoke({"city": city})
        if result.startswith("ERROR:"):
            raise ValueError(result)
        return result
    except Exception as e:
        return {"ok": False, "error": str(e), "action": "ask for a supported city"}

print("Call: weather_lookup('Mars')")
failure = safe_weather("Mars")
print("Observation:", failure)
print("Recovery: The agent should report that the city is unsupported and ask for another city.")


Call: weather_lookup('Mars')
Observation: {'ok': False, 'error': 'ERROR: unsupported city: Mars', 'action': 'ask for a supported city'}
Recovery: The agent should report that the city is unsupported and ask for another city.


## Failure modes and mitigations

| Failure mode | Mitigation |
|---|---|
| Wrong tool arguments | Strong schemas, precise docstrings, and validation |
| Tool throws an exception | Catch exceptions and return a clear error |
| Missing/unsupported data | Validate inputs and return a structured error |
| Agent loops too long | Set `max_iterations` / execution limits |
| Hallucinated tool choice | Restrict tools and make descriptions explicit |
| Free-form final output is hard to parse | Use Pydantic/structured output |

## Raw Python vs LangChain — short write-up

LangChain made tool registration, prompt composition, agent execution, message handling, and structured output much easier than the raw-Python implementation. Instead of manually maintaining every tool-call message and dispatching each function, the framework supplies reusable abstractions such as tools, prompts, agents, executors, and history wrappers. The abstraction leakiness is that debugging can become less transparent: some message formatting, tool routing, and intermediate state are handled inside the framework. Day 1 exposed the mechanics directly, while LangChain packages those mechanics into reusable components.

## Submission checklist

- [x] LangChain + langchain-anthropic setup
- [x] Raw Python → LangChain concept mapping
- [x] LCEL pipeline
- [x] 3 `@tool` tools
- [x] Real local JSON data source
- [x] Tool schemas and docstrings
- [x] AgentExecutor / tool-calling agent code
- [x] Annotated multi-step trace
- [x] Conversation memory
- [x] 3-turn follow-up scenario
- [x] Pydantic structured output
- [x] Tool failure + recovery
- [x] 6 failure modes + mitigations
- [x] Raw Python vs LangChain comparison
